## Check data subsetting
Use to evaluate how data is subsetting in xarray.

In [1]:
import xarray as xr
import pandas as pd
import os

In [2]:
# Load the SNOTEL summary CSV file
SNOTEL_PATH = "C:/Users/clmbn/NMT_PhD/data/snotel"
ESM1K_PATH = "C:/Users/clmbn/NMT_PhD/data/ESM1k_land"
snotel_file = os.path.join(SNOTEL_PATH, 'bcqc_data_v2', 'SNOTEL_summary.csv')
snotel_data = pd.read_csv(snotel_file,header=0)

In [3]:
netcdf_file = os.path.join(ESM1K_PATH,"ELEVATION_1k_c230606.nc")
elevation_ds = xr.open_dataset(netcdf_file)

In [4]:
elevation_ds

<xarray.Dataset> Size: 4GB
Dimensions:    (lat: 21600, lon: 43200)
Coordinates:
  * lat        (lat) float64 173kB -90.0 -89.99 -89.98 ... 89.98 89.99 90.0
  * lon        (lon) float64 346kB -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
Data variables:
    ELEVATION  (lat, lon) float32 4GB ...
Attributes:
    title:          Global 1km Land Surface Parameters for Kilometer Scale Ea...
    version:        1.0
    institution:    Atmospheric Sciences and Global Change Division, Pacific ...
    contact:        Lingcheng Li, lingcheng.li@pnnl.gov; Gautam Bisht, gautam...
    creation_date:  Tue Jan 16 22:48:54 PST 2024

In [13]:
# Check the first 5 rows of the SNOTEL data
five_sites = snotel_data.loc[:5, ['SNOTEL Name', 'SNOTEL Elev (ft)', 'SNOTEL Lat ()', 'SNOTEL Lon ()']]
print(five_sites.head(6))

     SNOTEL Name  SNOTEL Elev (ft)  SNOTEL Lat ()  SNOTEL Lon ()
0  Annie Springs              6010          42.87        -122.17
1      Long Lake               850          58.19        -133.83
2  Mcneil Canyon              1320          59.74        -151.25
3      Ute Creek             10650          37.61        -105.37
4     Lewis Peak              7400          40.36        -116.86
5     Onion Park              7410          46.91        -110.85


In [11]:
# Get elevation data for the first 5 SNOTEL sites
for index, row in five_sites.iterrows():
    site_name = row['SNOTEL Name']
    lat = row['SNOTEL Lat ()']
    lon = row['SNOTEL Lon ()']
    
    # Extract elevation data for the site
    elevation_value = elevation_ds.sel(lat=lat, lon=lon, method='nearest').ELEVATION.values * 3.28084  # Convert from meters to feet
    print(f"Site: {site_name}, Elevation: {elevation_value} ft")

Site: Annie Springs, Elevation: 6289.5 ft
Site: Long Lake, Elevation: 1358.6741943359375 ft
Site: Mcneil Canyon, Elevation: 1072.54052734375 ft
Site: Ute Creek, Elevation: 10658.8173828125 ft
Site: Lewis Peak, Elevation: 7470.08740234375 ft
Site: Onion Park, Elevation: 7496.85595703125 ft


In [15]:
# Generate a single subset dataset for five sites to compare with for loop
subset_ds = elevation_ds.sel(
    lat=xr.DataArray(five_sites['SNOTEL Lat ()'], dims="points"),
    lon=xr.DataArray(five_sites['SNOTEL Lon ()'], dims="points"),
    method="nearest"
)
subset_ds

<xarray.Dataset> Size: 168B
Dimensions:    (points: 6)
Coordinates:
    lat        (points) float64 48B 42.87 58.19 59.74 37.61 40.36 46.91
    lon        (points) float64 48B -122.2 -133.8 -151.2 -105.4 -116.9 -110.8
  * points     (points) int64 48B 0 1 2 3 4 5
Data variables:
    ELEVATION  (points) float32 24B ...
Attributes:
    title:          Global 1km Land Surface Parameters for Kilometer Scale Ea...
    version:        1.0
    institution:    Atmospheric Sciences and Global Change Division, Pacific ...
    contact:        Lingcheng Li, lingcheng.li@pnnl.gov; Gautam Bisht, gautam...
    creation_date:  Tue Jan 16 22:48:54 PST 2024